In [31]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import torch.nn.functional as F

In [2]:
words = open('names.txt','r',).read().splitlines()

building a bigram model
bigram -> only concerned with 2 characters at a time

In [ ]:
for w in words:
    for ch1, ch2 in zip(w,w[1:]):
        print(ch1,ch2)

In [6]:
w = words
list(w)
len(words)
#print(words[:10])

32033

In [ ]:
N = torch.zeros((27,27), dtype = torch.int32) # 26 alphabets+ E,S symbols



In [ ]:
#lookup table for all chars
# vocab of chars

chars =sorted(list(set(''.join(words)))) # set removes duplicates
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}




In [26]:
# building the dataset
# we use 3 characters to predict the next 4th character

block_size = 3 # context length
X, Y = [], []
for w in words[:5]:

    print(w)
    context = [0]*block_size
    for ch in w + '.':
        ix = stoi[ch]
       # print(ix)
       # print(context)
        X.append(context)
        Y.append(ix)

      #  print(X)
       # print(Y)
    
       
        print(''.join(itos[i] for i in context),'--->',itos[ix])
        context = context[1:] + [ix] # rolling context window
X = torch.tensor(X)
Y = torch.tensor(Y)


    


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [28]:
X.shape
X.dtype

torch.int64

In [36]:
C = torch.randn((27,2))# embeddings
C[5]

tensor([0.8938, 0.8149])

In [37]:
# one hot the embeddings
F.one_hot(torch.tensor(5),num_classes=27).float() @ C


tensor([0.8938, 0.8149])

In [41]:
emb =  C[X]
emb.shape

torch.Size([32, 3, 2])

In [ ]:
W1 = torch.randn((6,100)) # 100 neurons in the hidden layer
b1 = torch.randn(100)

In [54]:
h =torch.tanh( emb.view(-1,6) @ W1 +b1) # -1 forces torch to infer the shape iteslef according to 6 so its not hardocded
h

tensor([[ 0.9997, -0.4058,  0.9493,  ..., -0.9930,  0.8237,  0.8668],
        [ 0.9770, -0.9750,  0.1912,  ..., -0.9668,  0.6416,  0.9987],
        [ 0.9833, -0.0962,  0.9974,  ..., -0.6795,  0.9739, -0.9427],
        ...,
        [-0.9819,  0.2157,  0.7835,  ...,  0.9303,  0.9633, -0.9998],
        [-0.2019,  0.9833, -0.9297,  ..., -0.9996, -0.8901,  0.4098],
        [-0.9825, -0.7964,  0.8072,  ...,  0.9459,  0.9298, -0.9475]])

In [55]:
h.shape

torch.Size([32, 100])

In [56]:
W2 = torch.randn((100,27)) # 27 neurons in the output layer
b2 = torch.randn(27)

In [59]:
logits= h@W2 + b2
logits.shape

torch.Size([32, 27])

In [60]:
counts = logits.exp()
probs= counts/counts.sum(1,keepdims = True)

In [63]:

loss = -probs[torch.arange(32),Y].log().mean()     # probs assigned to the correct char by the nn

In [ ]:
g = torch.Generator().manual_seed(42)


In [ ]:
# end


In [ ]:

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1,ch2 in zip(chs,chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1,ix2] +=1

In [ ]:
itos = {i:s for s,i in stoi.items()}
itos

In [ ]:
plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i]+itos[j]
        plt.text(j,i,chstr,ha="center",va="bottom",color = "gray")
        plt.text(j,i,N[i,j].item(),ha = "center", va  = 'top',color = 'gray')
plt.axis('off')        


In [ ]:
P = (N+1).float()
P/= P.sum(1, keepdim =True)

### building a random prob distribution, then sampling from it


In [ ]:
g=torch.Generator().manual_seed(42)
p=torch.rand(3, generator=g) # 3 rand nos from 0 to 1
p=p/p.sum() # normalize the 3 rand nos to add up to 1
p # now p is a probability vector

In [ ]:
g=torch.Generator().manual_seed(42)
p = P[0]
ix = torch.multinomial(p,1,replacement =True,generator =g).item()# sample indices from p
itos[ix]

In [ ]:
torch.multinomial(p,20,replacement =True,generator =g)# sample indices from p


In [ ]:
g=torch.Generator().manual_seed(42)
for i in range(5):

    out = []
    ix = 0 # init index at <.>

    while True:
        p = P[ix]
       
        ix = torch.multinomial(p,1,replacement =True,generator =g).item()# sampling the next character from the distribution
        out.append(itos[ix])
    
        if ix==0: # after sampling the end token end the loop
            break  
    print(''.join(out))      


In [ ]:
log_likelihood = 0.0
n = 0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2] # prob of one correct bigram
        logprob= torch.log(prob)
        log_likelihood += logprob   # likellihood of a correct word = all prob multiplied, since we use log we add
        n +=1
        print(f'{ch1}{ch2}: {prob:.4f} {logprob: .4f}')
        nll = -log_likelihood # lower is better
print(f'{log_likelihood=}')  
print(f'{nll=}')
print(f'{nll/n=}')      

### creating training set of all the bigrams


In [ ]:
xs , ys = [],[] # inputs and targets
# input is the current char, target is the next char(sucessive)
for w in words:
    chs = ['.']+list(w)+['.']
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num =  xs.nelement()
print("number of exampels: ",num)

In [ ]:
xenc = F.one_hot(xs, num_classes = 27).float() # using one hot encoding to feed vectors to nn
xenc.shape
plt.imshow(xenc)

In [ ]:
# rand init of weights of 27 neurons
# each neuron 27 inputs

g = torch.Generator().manual_seed(42)
W = torch.randn((27,27),generator = g)


In [ ]:
logits = xenc @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims = True)

nlls = torch.zeros(5)
for i in range(5):
    x = xs[i].item()
    y = ys[i].item()

    print("--------")
    print(f"bigram example {i+1}:{itos[x]}{itos[y]}   (indexes {x}, {y})")
    print("input to the neural net: ",x)
    print("output prob from the neural net: ",probs[i])
    print("label (actual next character): ",y)
    p=probs[i,y]
    print("prob assigned to the correct char: ",p.item())
    logp = torch.log(p)
    print("log_likelihood",logp.item())
    nll = -logp
    print("negative log log_likelihood: ",nll.item())
    nlls[i] = nll
    print("---------")
    print("average negative log log_likelihood, ie loss",nlls.mean().item())



In [ ]:
# rand init of weights of 27 neurons
# each neuron 27 inputs

g = torch.Generator().manual_seed(42)
W = torch.randn((27,27),generator = g, requires_grad=True)


In [ ]:
# regularization 
(W**2).mean()

In [ ]:
# gradient descent
for k in range(100):
    # forwards pass

    xenc = F.one_hot(xs, num_classes = 27).float()
    logits = (xenc @ W)  # raw scores
    #softmax
    counts = logits.exp()# counts are now positive log counts
    probs = counts / counts.sum(1, keepdims = True)  # normalize the counts to prob distri.
    loss =  -probs[torch.arange(num),ys].log().mean() + 0.01*(W**2).mean()# for only first 5 birgrams -> first words ".emma."
    print(loss.item())

        # backward pass
    W.grad = None # set grad to 0 for init
    loss.backward()

    # update W
    W.data += -50*W.grad





In [ ]:
# sampling from neural net model
g = torch.Generator().manual_seed(42)
for i in range(5):
     out = []
     ix = 0
     while True:
         xenc = F.one_hot(torch.tensor([ix]), num_classes = 27).float()
         logits = xenc @ W
         counts = logits.exp()
         p = counts / counts.sum(1, keepdims = True)
         ix = torch.multinomial(p[0],num_samples = 1,replacement = True, generator=g).item()
         out.append(itos[ix])
         if ix ==0:
            break
     print(''.join(out))